# Phase 1 — Lesson 20: The Fourier Transform
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 20: The Fourier Transform
**Type:** Build | **Languages:** Python | **Time:** ~90 minutes
**Prerequisites:** Phase 1, Lessons 01-04, 19



# What You'll Learn

---

## Learning Objectives

- Implement the DFT from scratch and verify it against the O(N log N) Cooley-Tukey FFT
- Interpret frequency coefficients: extract amplitude, phase, and power spectrum
- Apply the convolution theorem to perform convolution via FFT multiplication
- Connect Fourier frequency decomposition to transformer positional encodings and CNN convolution layers


# What Is the Fourier Transform?

---

Any signal can be decomposed into a sum of sine and cosine waves at different frequencies.

```
X[k] = Σ_{n=0}^{N−1} x[n] · e^(−2πi·k·n/N)
```

X[k] is a complex number. |X[k]| = amplitude of frequency k. arg(X[k]) = phase.


In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport cmathimport mathdef dft(x):    N = len(x)    return np.array([sum(x[n] * cmath.exp(-2j * math.pi * k * n / N) for n in range(N)) for k in range(N)])N = 64n = np.arange(N)x = np.sin(2 * np.pi * 2 * n / N) + 0.5 * np.sin(2 * np.pi * 5 * n / N)X = dft(x)magnitudes = np.abs(X)peaks = np.argsort(magnitudes)[::-1][:5]print(f"Top frequency bins: {peaks}")print(f"Magnitude at bin 2: {magnitudes[2]:.2f}")print(f"Magnitude at bin 5: {magnitudes[5]:.2f}")plt.stem(n[:N//2], magnitudes[:N//2], linefmt='#2c3e50', markerfmt=' ', basefmt=' ')plt.title('DFT Magnitude Spectrum (peaks at bins 2 and 5)', fontsize=12, fontweight='bold')plt.xlabel('Frequency bin'); plt.ylabel('|X[k]|')plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# DFT vs FFT

---

**DFT:** O(N²) direct computation.
**FFT:** Cooley-Tukey algorithm, O(N log N) — recursive divide and conquer.


In [ ]:
import cmathimport mathimport numpy as npdef fft(x):    N = len(x)    if N == 1:        return x    even = fft(x[0::2])    odd = fft(x[1::2])    X = [0j] * N    for k in range(N // 2):        twiddle = cmath.exp(-2j * math.pi * k / N) * odd[k]        X[k] = even[k] + twiddle        X[k + N // 2] = even[k] - twiddle    return Xx = np.sin(2 * np.pi * 2 * np.arange(64) / 64) + 0.5 * np.sin(2 * np.pi * 5 * np.arange(64) / 64)X_fft = fft(x.tolist())X_dft = dft(x.tolist())print(f"FFT vs DFT max diff: {max(abs(X_fft[k] - X_dft[k]) for k in range(64)):.2e}")

# Inverse DFT

---

Reconstruct the signal from frequency coefficients:

```
x[n] = (1/N) Σ X[k] · e^(2πi·k·n/N)
```

Perfect reconstruction — no information is lost.


In [ ]:
import numpy as npimport cmathimport mathdef idft(X):    N = len(X)    return np.array([sum(X[k] * cmath.exp(2j * math.pi * k * n / N) for k in range(N)) / N for n in range(N)])N = 64n = np.arange(N)x = np.sin(2 * np.pi * 2 * n / N) + 0.5 * np.sin(2 * np.pi * 5 * n / N)X = dft(x)x_recon = np.real(idft(X))err = np.max(np.abs(x - x_recon))print(f"Max reconstruction error: {err:.2e}")print("Perfect reconstruction — DFT is a change of basis.")

# Power Spectrum and Spectral Leakage

---

|X[k]|² = power at frequency k.

Windowing reduces spectral leakage by tapering the signal to zero at both ends.


In [ ]:
import numpy as npimport matplotlib.pyplot as pltN = 256n = np.arange(N)# Signal with frequency not on a bin (causes leakage)x = np.sin(2 * np.pi * 10.5 * n / N)X_rect = np.abs(np.fft.fft(x))X_hann = np.abs(np.fft.fft(x * np.hanning(N)))freqs = np.fft.fftfreq(N)[:N//2]plt.plot(freqs, X_rect[:N//2], label='Rectangular', alpha=0.7)plt.plot(freqs, X_hann[:N//2], label='Hann window', alpha=0.7)plt.title('Spectral Leakage: Windowing Helps', fontsize=12, fontweight='bold')plt.xlabel('Frequency (cycles/sample)'); plt.ylabel('|X[k]|')plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Convolution Theorem

---

Convolution in time = multiplication in frequency:

```
conv(x, h) = IFFT(FFT(x) · FFT(h))
```

This is what happens in convolutional layers with large kernels.


In [ ]:
import numpy as npx = np.array([1, 2, 3, 4])h = np.array([1, 1, 1])# Direct convolutionconv_direct = np.convolve(x, h)# FFT-based convolutionN = len(x) + len(h) - 1N_pad = 1while N_pad < N:    N_pad *= 2X = np.fft.fft(np.pad(x, (0, N_pad - len(x))))H = np.fft.fft(np.pad(h, (0, N_pad - len(h))))conv_fft = np.fft.ifft(X * H)[:N].realprint(f"Direct:  {conv_direct}")print(f"FFT:     {np.round(conv_fft, 6)}")print(f"Match: {np.allclose(conv_direct, conv_fft)}")

# Positional Encodings and CNNs

---

**Positional encodings:** sin/cos at geometrically spaced frequencies — each position gets a unique frequency fingerprint.

**CNNs:** convolution in time = multiplication in frequency — FFT enables fast convolution for large kernels.


In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Positional encoding dot product depends only on relative positiondef pos_enc(pos, d=64):    i = np.arange(d)[None, :]    rates = 1 / np.power(10000, (2 * (i // 2)) / d)    rads = pos * rates    rads[:, 0::2] = np.sin(rads[:, 0::2])    rads[:, 1::2] = np.cos(rads[:, 1::2])    return rads[0]pe_0 = pos_enc(0)pe_5 = pos_enc(5)pe_10 = pos_enc(10)pe_15 = pos_enc(15)print(f"dot(PE(0), PE(5)) = {np.dot(pe_0, pe_5):.4f}")print(f"dot(PE(10), PE(15)) = {np.dot(pe_10, pe_15):.4f}")print("(Same relative distance → same dot product)")

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** What does |X[k]| represent?
> **Explanation:** Amplitude of frequency bin k — how much of that frequency is in the signal.

**Q2:** What is the convolution theorem?
> **Explanation:** Convolution in time = pointwise multiplication in frequency — enables fast convolution via FFT.

**Q3:** What does zero-padding do?
> **Explanation:** Interpolates between frequency bins — does NOT improve true resolution.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Verify DFT of a pure sine wave has peaks at ±frequency.

**Exercise 2:** Implement a low-pass filter: zero high-frequency bins, inverse DFT, verify smoothing.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| DFT | Discrete Fourier Transform | Correlates signal with N complex exponentials — O(N²) |
| FFT | Fast Fourier Transform | Cooley-Tukey O(N log N) algorithm |
| Frequency bin | Index k | Represents frequency k·fs/N — amplitude |X[k]|, phase arg(X[k]) |
| Power spectrum | |X[k]|² | Energy at each frequency |
| Convolution theorem | conv = IFFT(FFT·FFT) | Convolution in time = multiplication in frequency |
| Spectral leakage | Spurious frequencies | Caused by treating non-periodic signal as periodic — fixed by windowing |
| Twiddle factor | e^(−2πi·k/N) | Rotation used in FFT butterfly operations |


# Further Reading

---

- [3Blue1Brown: But what is the Fourier Transform?](https://www.youtube.com/watch?v=spUNpyF58BY)
- [FNet: Mixing Tokens with Fourier Transforms](https://arxiv.org/abs/2105.03824)
- [Smith: DSP Guide](http://www.dspguide.com/)


# Next Steps

---

When you're ready, continue to Lesson 21: Graph Theory for Machine Learning.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
